# GPU RF Shared-Book ML Trading

Train collapsed-label CUDA random forests on all available pre-2020 Quant Warehouse data, then trade the 2020+ out-of-sample predictions with one shared multi-asset book.

The strategy does not rebalance to top-k every day. It holds positions until an opposite signal exits them, then fills open slots from the best current opportunities that pass the `> 0.5` score filter. Position size is fixed at `1 / top_k`, so unused slots remain cash.

The notebook compares long-only, short-only, and shared long+short variants for `top_k = [5, 10, 20, 40]`. Each variant is tested with the ensemble mean score and with every individual feature-family ML model as its own strategy source. Execution uses native Zipline multi-asset shared-book orders through `order_target_percent`.


In [1]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import random
import sys
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, f1_score
from sklearn.preprocessing import LabelEncoder

QUANT_WAREHOUSE_ROOT = Path('/home/jlee153232/PycharmProjects/quant-warehouse')
if str(QUANT_WAREHOUSE_ROOT) not in sys.path:
    sys.path.insert(0, str(QUANT_WAREHOUSE_ROOT))

import cupy as cp
import cudf
from cuml.ensemble import RandomForestClassifier as CuRandomForestClassifier

from quant_orchestrator.platforms.backtesting_frameworks.shared_book import build_shared_book_weights
from quant_orchestrator.platforms.backtesting_frameworks.zipline.shared_book import (
    ZiplineSharedBookSummaryJob,
    run_zipline_shared_book_summary_jobs,
)

from quant_warehouse.platforms.data_providers.fmp.target_engineering.event_pairs import EventPairStore
from quant_warehouse.research_tools import (
    BinaryTargetConfig,
    FamilyEvaluationConfig,
    build_collapsed_bullish_event_target_panel,
    build_fundamental_feature_panel,
    build_oracle_trade_target_panel,
    cap_features_by_quality,
    load_fmp_event_pairs,
    screen_fmp_equity_universe,
)
from quant_warehouse.warehouse.api import Warehouse

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')
warnings.filterwarnings('ignore', category=FutureWarning)

print('cupy', cp.__version__, 'cuda_devices', cp.cuda.runtime.getDeviceCount())
print('cudf', cudf.__version__)

cupy 14.1.1 cuda_devices 1
cudf 26.06.00


In [2]:
RANDOM_SEED = 20260702
MIN_MARKET_CAP = 1_000_000_000_000
START_DATE = '1900-01-01'
END_DATE = None
TRAIN_END = pd.Timestamp('2019-12-31')
OOS_START = pd.Timestamp('2020-01-01')

TOP_K_VALUES = [5, 10, 20, 40]
ENTRY_THRESHOLD = 0.50
EXIT_THRESHOLD = 0.50
MIN_FEATURE_COVERAGE = 0.50
MAX_FEATURES_PER_FAMILY = 50
MIN_TRAIN_ROWS_PER_FAMILY = 250
MIN_CLASSES_PER_FAMILY = 2
CAPITAL_BASE = 1_000_000.0

ZIPLINE_COMMISSION_PER_SHARE = 0.005
ZIPLINE_SLIPPAGE_BPS = 5.0
ZIPLINE_MAX_WORKERS = 4

RF_PARAMS = {
    'n_estimators': 300,
    'max_depth': 16,
    'max_features': 'sqrt',
    'n_bins': 128,
    'random_state': RANDOM_SEED,
    'n_streams': 8,
}

FEATURE_CONFIG = FamilyEvaluationConfig(
    provider='fmp',
    market_cap_min=MIN_MARKET_CAP,
    start_date=START_DATE,
    end_date=END_DATE,
    max_features_per_family=MAX_FEATURES_PER_FAMILY,
)
TARGET_CONFIG = BinaryTargetConfig(
    provider='fmp',
    start_date=START_DATE,
    end_date=END_DATE,
    event_families=('congress', 'insider', 'analyst_rating', 'price_target', 'earnings'),
    oracle_trade_k_by_frequency={'YE': tuple(range(1, 13))},
)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
FEATURE_CONFIG, TARGET_CONFIG

(FamilyEvaluationConfig(provider='fmp', market_cap_min=1000000000000, country='US', exchanges=('NASDAQ', 'NYSE', 'AMEX'), screen_limit=5000, start_date='1900-01-01', end_date=None, filing_lag_days=45, horizons=(20, 60, 120), min_observations=120, max_features_per_family=50),
 BinaryTargetConfig(provider='fmp', start_date='1900-01-01', end_date=None, event_families=('congress', 'insider', 'analyst_rating', 'price_target', 'earnings'), oracle_trade_k_by_frequency={'YE': (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12)}, oracle_trade_min_profit_pct=0.01, oracle_trade_long_entry_price_col='high', oracle_trade_long_exit_price_col='low', oracle_trade_short_entry_price_col='low', oracle_trade_short_exit_price_col='high', event_alignment_tolerance_days=7, collapsed_bullish_event_types=('congress_buy', 'insider_buy', 'analyst_upgrade', 'price_target_raise', 'earnings_beat')))

## Build Data

Training uses every available row before 2020. Out-of-sample classification metrics and trading results both use 2020+ rows.

In [3]:
started = perf_counter()
WAREHOUSE = Warehouse()
EVENT_STORE = EventPairStore(backend=WAREHOUSE.backend, catalog=WAREHOUSE.catalog)

symbols, raw_universe, universe_eligibility, universe_source = screen_fmp_equity_universe(FEATURE_CONFIG, warehouse=WAREHOUSE)
print({'universe_source': universe_source, 'eligible_symbols': len(symbols)})
display(universe_eligibility.loc[universe_eligibility['eligible']].head(30))

{'universe_source': 'openbb:fmp', 'eligible_symbols': 14}


,symbol,eligible,reason,screen_market_cap
0,NVDA,True,ok,4785585180000
1,GOOGL,True,ok,4368788098535
2,GOOG,True,ok,4349493623926
3,AAPL,True,ok,4323663859280
4,MSFT,True,ok,2854597080400
5,AMZN,True,ok,2599991070000
6,SPCX,True,ok,2059971841733
7,AVGO,True,ok,1757164597200
8,TSLA,True,ok,1597307716000
9,META,True,ok,1555825021126


In [4]:
raw_feature_panel, raw_feature_metadata, feature_diagnostics, feature_timings = build_fundamental_feature_panel(
    symbols, FEATURE_CONFIG, warehouse=WAREHOUSE
)
selected_features, selected_feature_metadata, feature_quality = cap_features_by_quality(
    raw_feature_panel, raw_feature_metadata, max_features=MAX_FEATURES_PER_FAMILY
)
feature_panel = raw_feature_panel[['symbol', 'date', *selected_features]].copy()
feature_panel['symbol'] = feature_panel['symbol'].astype(str).str.upper()
feature_panel['date'] = pd.to_datetime(feature_panel['date'], errors='coerce').dt.normalize()
print({
    'raw_feature_panel_rows': len(raw_feature_panel),
    'selected_feature_columns': len(selected_features),
    'selected_feature_families': selected_feature_metadata['family'].nunique(),
    **feature_timings,
})
display(selected_feature_metadata.groupby(['source', 'family'], as_index=False).agg(feature_count=('feature', 'nunique')).sort_values(['source', 'family']))

{'raw_feature_panel_rows': 100125, 'selected_feature_columns': 365, 'selected_feature_families': 15, 'raw_panel_build_seconds': 1.5188049799762666}


,source,family,feature_count
0,financetoolkit,ft_growth_balance,50
1,financetoolkit,ft_growth_cash,50
2,financetoolkit,ft_growth_income,38
3,financetoolkit,ft_ratios_efficiency,5
4,financetoolkit,ft_ratios_liquidity,7
5,financetoolkit,ft_ratios_profitability,14
6,financetoolkit,ft_ratios_solvency,15
7,financetoolkit,ft_ratios_valuation,24
8,fmp,fmp_balance_mcap,50
9,fmp,fmp_cash_mcap,39


In [5]:
events, event_diagnostics, event_load_seconds = load_fmp_event_pairs(
    symbols, TARGET_CONFIG, event_store=EVENT_STORE, include_historical=True
)
event_symbols = tuple(event_diagnostics.loc[event_diagnostics['combined_rows'].gt(0), 'symbol'].sort_values())
feature_panel = feature_panel.loc[feature_panel['symbol'].isin(event_symbols)].copy()

collapsed_event_panel, collapsed_event_metadata = build_collapsed_bullish_event_target_panel(
    feature_panel[['symbol', 'date']], events, TARGET_CONFIG
)
oracle_panel, oracle_metadata, oracle_seconds = build_oracle_trade_target_panel(
    event_symbols, TARGET_CONFIG, warehouse=WAREHOUSE
)
print({
    'event_symbols': len(event_symbols),
    'event_rows': len(events),
    'event_load_seconds': round(event_load_seconds, 3),
    'collapsed_event_rows': len(collapsed_event_panel),
    'oracle_rows': len(oracle_panel),
    'oracle_columns': len(oracle_metadata),
    'oracle_seconds': round(oracle_seconds, 3),
})
display(event_diagnostics.sort_values('combined_rows', ascending=False).head(20))

{'event_symbols': 13, 'event_rows': 5836, 'event_load_seconds': 0.859, 'collapsed_event_rows': 100114, 'oracle_rows': 100114, 'oracle_columns': 36, 'oracle_seconds': 4.091}


,symbol,cached_rows,historical_rows,combined_rows,event_families
9,MSFT,65,971,1036,"(analyst_rating, congress, earnings, insider, ..."
0,AAPL,62,885,947,"(analyst_rating, congress, earnings, insider, ..."
11,NVDA,47,812,859,"(analyst_rating, congress, earnings, insider, ..."
1,AMZN,66,742,808,"(analyst_rating, congress, earnings, insider, ..."
6,GOOGL,60,590,650,"(analyst_rating, congress, earnings, insider, ..."
13,TSLA,60,459,519,"(analyst_rating, congress, earnings, insider, ..."
8,META,55,437,492,"(analyst_rating, congress, earnings, insider, ..."
10,MU,0,121,121,"(earnings,)"
7,LLY,0,104,104,"(earnings,)"
3,BRK-A,0,102,102,"(earnings,)"


In [6]:
def collapsed_label_rows(collapsed_event_panel: pd.DataFrame, oracle_panel: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    event_target = 'target_event_collapsed__bullish'
    event_activity = f'_target_activity__{event_target}'
    frames = []
    diagnostics = []

    event_frame = collapsed_event_panel.copy()
    if event_target in event_frame.columns:
        activity = pd.to_numeric(event_frame.get(event_activity, 0), errors='coerce').fillna(0).astype('int8')
        value = pd.to_numeric(event_frame[event_target], errors='coerce').fillna(0).astype('int8')
        bullish = event_frame.loc[value.eq(1), ['symbol', 'date']].copy()
        bullish['collapsed_label'] = 'event_bullish'
        bullish['label_source'] = 'event_collapsed'
        frames.append(bullish)
        diagnostics.append({'source': 'event_collapsed', 'candidate_rows': int(activity.gt(0).sum()), 'used_rows': len(bullish), 'dropped_rows': int(activity.gt(0).sum()) - len(bullish), 'note': 'mirror/non-bullish event rows excluded'})

    long_cols = sorted(c for c in oracle_panel.columns if c.startswith('target_oracle_trade_entry__') and c.endswith('_long'))
    short_cols = sorted(c for c in oracle_panel.columns if c.startswith('target_oracle_trade_entry__') and c.endswith('_short'))
    if long_cols and short_cols:
        oracle = oracle_panel[['symbol', 'date', *long_cols, *short_cols]].copy()
        long_any = oracle[long_cols].apply(pd.to_numeric, errors='coerce').fillna(0).gt(0).any(axis=1)
        short_any = oracle[short_cols].apply(pd.to_numeric, errors='coerce').fillna(0).gt(0).any(axis=1)
        ambiguous = long_any & short_any
        long_rows = oracle.loc[long_any & ~short_any, ['symbol', 'date']].copy()
        long_rows['collapsed_label'] = 'oracle_long'
        long_rows['label_source'] = 'oracle_trade'
        short_rows = oracle.loc[short_any & ~long_any, ['symbol', 'date']].copy()
        short_rows['collapsed_label'] = 'oracle_short'
        short_rows['label_source'] = 'oracle_trade'
        frames.extend([long_rows, short_rows])
        diagnostics.append({'source': 'oracle_trade', 'candidate_rows': int((long_any | short_any).sum()), 'used_rows': len(long_rows) + len(short_rows), 'dropped_rows': int(ambiguous.sum()), 'note': 'ambiguous long+short rows dropped after k collapse'})

    labels = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=['symbol', 'date', 'collapsed_label', 'label_source'])
    labels['symbol'] = labels['symbol'].astype(str).str.upper()
    labels['date'] = pd.to_datetime(labels['date'], errors='coerce').dt.normalize()
    labels = labels.dropna(subset=['symbol', 'date', 'collapsed_label']).drop_duplicates()
    return labels.sort_values(['date', 'symbol', 'collapsed_label']).reset_index(drop=True), pd.DataFrame(diagnostics)

label_rows, label_diagnostics = collapsed_label_rows(collapsed_event_panel, oracle_panel)
print({'collapsed_label_rows': len(label_rows), 'classes': sorted(label_rows['collapsed_label'].unique())})
display(label_diagnostics)
display(label_rows.groupby(['label_source', 'collapsed_label'], as_index=False).agg(rows=('symbol', 'size'), symbols=('symbol', 'nunique'), min_date=('date', 'min'), max_date=('date', 'max')))

{'collapsed_label_rows': 11581, 'classes': ['event_bullish', 'oracle_long', 'oracle_short']}


,source,candidate_rows,used_rows,dropped_rows,note
0,event_collapsed,4131,2353,1778,mirror/non-bullish event rows excluded
1,oracle_trade,9228,9228,0,ambiguous long+short rows dropped after k coll...


,label_source,collapsed_label,rows,symbols,min_date,max_date
0,event_collapsed,event_bullish,2353,13,1993-03-26,2026-06-22
1,oracle_trade,oracle_long,4677,13,1970-07-17,2026-06-12
2,oracle_trade,oracle_short,4551,13,1970-07-09,2026-06-22


## Train GPU RF Family Models Pre-2020

Each feature family gets one CUDA random forest. Out-of-sample metrics are measured on 2020+ labeled rows.

In [7]:
def prepare_family_dataset(feature_panel, feature_metadata, labels, *, source, family, min_feature_coverage):
    family_meta = feature_metadata.loc[feature_metadata['source'].astype(str).eq(source) & feature_metadata['family'].astype(str).eq(family)]
    features = [f for f in family_meta['feature'].drop_duplicates().tolist() if f in feature_panel.columns]
    numeric_features = [f for f in features if pd.to_numeric(feature_panel[f], errors='coerce').notna().any()]
    if not numeric_features:
        return pd.DataFrame(), []
    merged = labels.merge(feature_panel[['symbol', 'date', *numeric_features]], on=['symbol', 'date'], how='inner')
    if merged.empty:
        return merged, numeric_features
    numeric = merged[numeric_features].apply(pd.to_numeric, errors='coerce')
    coverage = numeric.notna().mean(axis=1)
    merged = merged.loc[coverage.ge(min_feature_coverage)].copy()
    if merged.empty:
        return merged, numeric_features
    numeric = numeric.loc[merged.index]
    medians = numeric.median(axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    merged[numeric_features] = numeric.replace([np.inf, -np.inf], np.nan).fillna(medians).astype('float32')
    return merged.reset_index(drop=True), numeric_features

def fit_gpu_random_forest(train, features):
    encoder = LabelEncoder()
    y = encoder.fit_transform(train['collapsed_label'].astype(str))
    model = CuRandomForestClassifier(**RF_PARAMS)
    model.fit(cudf.from_pandas(train[features].astype('float32')), cudf.Series(y.astype('int32')))
    return model, encoder

def predict_proba_frame(model, encoder, frame, features):
    proba = model.predict_proba(cudf.from_pandas(frame[features].astype('float32')))
    proba_np = proba.to_numpy() if hasattr(proba, 'to_numpy') else cp.asnumpy(proba)
    out = pd.DataFrame(proba_np, columns=[f'prob__{label}' for label in encoder.classes_], index=frame.index)
    for label in ['event_bullish', 'oracle_long', 'oracle_short']:
        col = f'prob__{label}'
        if col not in out.columns:
            out[col] = 0.0
    return out

def score_classifier(model, encoder, frame, features):
    proba = predict_proba_frame(model, encoder, frame, features)
    y_true = frame['collapsed_label'].astype(str).to_numpy()
    y_pred = encoder.inverse_transform(np.asarray(proba[[f'prob__{label}' for label in encoder.classes_]].to_numpy().argmax(axis=1)).astype(int))
    return {
        'rows': len(frame),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }

model_rows = []
models = {}
train_started = perf_counter()
for source, family in selected_feature_metadata[['source', 'family']].drop_duplicates().sort_values(['source', 'family']).itertuples(index=False, name=None):
    family_frame, features = prepare_family_dataset(feature_panel, selected_feature_metadata, label_rows, source=str(source), family=str(family), min_feature_coverage=MIN_FEATURE_COVERAGE)
    if family_frame.empty:
        model_rows.append({'source': source, 'family': family, 'status': 'skipped_empty', 'features': len(features), 'rows': 0})
        continue
    train = family_frame.loc[pd.to_datetime(family_frame['date']).le(TRAIN_END)].copy()
    oos = family_frame.loc[pd.to_datetime(family_frame['date']).ge(OOS_START)].copy()
    if len(train) < MIN_TRAIN_ROWS_PER_FAMILY or train['collapsed_label'].nunique() < MIN_CLASSES_PER_FAMILY:
        model_rows.append({'source': source, 'family': family, 'status': 'skipped_sparse_train', 'features': len(features), 'rows': len(family_frame), 'train_rows': len(train), 'oos_rows': len(oos), 'train_classes': train['collapsed_label'].nunique()})
        continue
    fit_started = perf_counter()
    model, encoder = fit_gpu_random_forest(train, features)
    fit_seconds = perf_counter() - fit_started
    models[(source, family)] = {'model': model, 'encoder': encoder, 'features': features}
    train_scores = score_classifier(model, encoder, train, features)
    oos_scores = score_classifier(model, encoder, oos, features) if not oos.empty and oos['collapsed_label'].nunique() > 1 else {'rows': len(oos), 'accuracy': np.nan, 'balanced_accuracy': np.nan, 'macro_f1': np.nan}
    model_rows.append({'source': source, 'family': family, 'status': 'ok', 'features': len(features), 'rows': len(family_frame), 'train_rows': len(train), 'oos_rows': len(oos), 'classes': family_frame['collapsed_label'].nunique(), 'fit_seconds': fit_seconds, **{f'train_{k}': v for k, v in train_scores.items()}, **{f'oos_{k}': v for k, v in oos_scores.items()}})

model_results = pd.DataFrame(model_rows).sort_values(['status', 'oos_macro_f1', 'oos_balanced_accuracy'], ascending=[True, False, False]).reset_index(drop=True)
print({'trained_models': len(models), 'elapsed_seconds': round(perf_counter() - train_started, 3)})
display(model_results)

[21:50:29] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[21:50:31] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[21:50:33] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[21:50:34] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[21:50:36] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


...

[21:50:51] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[21:50:52] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


{'trained_models': 15, 'elapsed_seconds': 27.418}


[21:50:54] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


,source,family,status,features,rows,train_rows,oos_rows,classes,fit_seconds,train_accuracy,train_balanced_accuracy,train_macro_f1,oos_accuracy,oos_balanced_accuracy,oos_macro_f1
0,financetoolkit,ft_ratios_valuation,ok,24,10885,7126,3759,3,1.5384,0.5265,0.4494,0.4688,0.4302,0.3953,0.3912
1,financetoolkit,ft_ratios_solvency,ok,15,10885,7126,3759,3,1.4999,0.5236,0.4464,0.4660,0.2934,0.3448,0.2676
2,fmp,fmp_cash_mcap,ok,39,10430,6671,3759,3,1.5578,0.9319,0.8437,0.8741,0.2969,0.3642,0.2555
3,financetoolkit,ft_ratios_efficiency,ok,5,10885,7126,3759,3,1.6000,0.5240,0.4456,0.4654,0.2854,0.3410,0.2515
4,financetoolkit,ft_ratios_profitability,ok,14,10885,7126,3759,3,1.6028,0.5261,0.4464,0.4659,0.2825,0.3402,0.2496
5,fmp,fmp_daily_ev_yield,ok,7,10885,7126,3759,3,1.6300,0.9145,0.8062,0.8417,0.2876,0.3533,0.2486
6,fmp,fmp_daily_mcap_yield,ok,14,10826,7067,3759,3,1.6579,0.9530,0.8749,0.9026,0.2961,0.3635,0.2480
7,fmp,fmp_balance_mcap,ok,50,10826,7067,3759,3,1.6189,0.9458,0.8676,0.8971,0.2985,0.3607,0.2423
8,fmp,fmp_income_mcap,ok,31,10885,7126,3759,3,1.6287,0.9499,0.8769,0.9026,0.2966,0.3588,0.2413
9,fmp,fmp_daily_ev_multiple,ok,7,10822,7063,3759,3,1.5807,0.9216,0.8145,0.8504,0.2857,0.3524,0.2413


## Inference Scores

Scores are averaged across feature-family models. `long_score = P(event_bullish) + P(oracle_long)`, capped at 1. `short_score = P(oracle_short)`.

In [8]:
def prepare_prediction_frame(feature_panel, features, min_feature_coverage):
    base_cols = ['symbol', 'date']
    numeric = feature_panel[features].apply(pd.to_numeric, errors='coerce')
    coverage = numeric.notna().mean(axis=1)
    out = feature_panel.loc[coverage.ge(min_feature_coverage), base_cols].copy()
    if out.empty:
        return out
    numeric = numeric.loc[out.index]
    medians = numeric.median(axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    out[features] = numeric.replace([np.inf, -np.inf], np.nan).fillna(medians).astype('float32')
    return out.reset_index(drop=True)


def strategy_source_name(source, family):
    return f'{source}.{family}'


pred_frames = []
for (source, family), payload in models.items():
    pred_input = prepare_prediction_frame(feature_panel, payload['features'], MIN_FEATURE_COVERAGE)
    if pred_input.empty:
        continue
    proba = predict_proba_frame(payload['model'], payload['encoder'], pred_input, payload['features'])
    scored = pred_input[['symbol', 'date']].copy()
    scored['source'] = str(source)
    scored['family'] = str(family)
    scored['strategy_source'] = strategy_source_name(source, family)
    scored['long_score_component'] = (proba['prob__event_bullish'] + proba['prob__oracle_long']).clip(0, 1).to_numpy()
    scored['short_score_component'] = proba['prob__oracle_short'].clip(0, 1).to_numpy()
    pred_frames.append(scored)

predictions_by_family = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()

mean_scores = (
    predictions_by_family.groupby(['symbol', 'date'], as_index=False)
    .agg(long_score=('long_score_component', 'mean'), short_score=('short_score_component', 'mean'), model_count=('strategy_source', 'nunique'))
)
mean_scores = mean_scores.loc[pd.to_datetime(mean_scores['date']).ge(OOS_START)].copy()
mean_scores['source'] = 'ensemble'
mean_scores['family'] = 'mean'
mean_scores['strategy_source'] = 'ensemble_mean'
mean_scores['net_score'] = mean_scores['long_score'] - mean_scores['short_score']

single_model_scores = predictions_by_family.rename(
    columns={'long_score_component': 'long_score', 'short_score_component': 'short_score'}
).copy()
single_model_scores = single_model_scores.loc[pd.to_datetime(single_model_scores['date']).ge(OOS_START)].copy()
single_model_scores['model_count'] = 1
single_model_scores['net_score'] = single_model_scores['long_score'] - single_model_scores['short_score']
single_model_scores = single_model_scores[
    ['strategy_source', 'source', 'family', 'symbol', 'date', 'long_score', 'short_score', 'net_score', 'model_count']
]

strategy_scores = pd.concat(
    [
        mean_scores[['strategy_source', 'source', 'family', 'symbol', 'date', 'long_score', 'short_score', 'net_score', 'model_count']],
        single_model_scores,
    ],
    ignore_index=True,
)

print({
    'ensemble_prediction_rows': len(mean_scores),
    'single_model_prediction_rows': len(single_model_scores),
    'strategy_sources': strategy_scores['strategy_source'].nunique(),
    'symbols': strategy_scores['symbol'].nunique(),
    'dates': strategy_scores['date'].nunique(),
})
display(mean_scores[['long_score', 'short_score', 'net_score', 'model_count']].describe())
display(
    strategy_scores.groupby(['strategy_source', 'source', 'family'], as_index=False)
    .agg(rows=('symbol', 'size'), symbols=('symbol', 'nunique'), dates=('date', 'nunique'))
    .sort_values(['strategy_source'])
)


{'ensemble_prediction_rows': 21151, 'single_model_prediction_rows': 317265, 'strategy_sources': 16, 'symbols': 13, 'dates': 1627}


,long_score,short_score,net_score,model_count
count,"21,151.0000","21,151.0000","21,151.0000","21,151.0000"
mean,0.5349,0.4651,0.0698,15.0000
std,0.0564,0.0564,0.1128,0.0000
min,0.2304,0.1793,-0.5392,15.0000
25%,0.5039,0.4327,0.0078,15.0000
50%,0.5405,0.4595,0.0811,15.0000
75%,0.5673,0.4961,0.1346,15.0000
max,0.8207,0.7696,0.6413,15.0000


,strategy_source,source,family,rows,symbols,dates
0,ensemble_mean,ensemble,mean,21151,13,1627
1,financetoolkit.ft_growth_balance,financetoolkit,ft_growth_balance,21151,13,1627
2,financetoolkit.ft_growth_cash,financetoolkit,ft_growth_cash,21151,13,1627
3,financetoolkit.ft_growth_income,financetoolkit,ft_growth_income,21151,13,1627
4,financetoolkit.ft_ratios_efficiency,financetoolkit,ft_ratios_efficiency,21151,13,1627
5,financetoolkit.ft_ratios_liquidity,financetoolkit,ft_ratios_liquidity,21151,13,1627
6,financetoolkit.ft_ratios_profitability,financetoolkit,ft_ratios_profitability,21151,13,1627
7,financetoolkit.ft_ratios_solvency,financetoolkit,ft_ratios_solvency,21151,13,1627
8,financetoolkit.ft_ratios_valuation,financetoolkit,ft_ratios_valuation,21151,13,1627
9,fmp.fmp_balance_mcap,fmp,fmp_balance_mcap,21151,13,1627


In [9]:
def load_price_frames(symbols):
    frames = {}
    for symbol in symbols:
        prices = WAREHOUSE.read_prices(symbol, provider=FEATURE_CONFIG.provider, start=START_DATE, end=END_DATE)
        if prices is None or prices.empty:
            continue
        frame = prices.rename(columns=str.lower).copy()
        required = ['open', 'high', 'low', 'close', 'volume']
        if not set(required).issubset(frame.columns):
            continue
        frame = frame[required].apply(pd.to_numeric, errors='coerce')
        frame.index = pd.DatetimeIndex(pd.to_datetime(frame.index, errors='coerce')).normalize()
        frame = frame.dropna(subset=['open', 'high', 'low', 'close']).sort_index()
        if not frame.empty:
            frames[str(symbol).upper()] = frame
    return frames

price_frames = load_price_frames(event_symbols)
wide_close = pd.DataFrame({symbol: frame['close'] for symbol, frame in price_frames.items()}).sort_index().ffill()
next_returns = wide_close.pct_change().shift(-1)
print({'price_symbols': wide_close.shape[1], 'price_dates': wide_close.shape[0], 'oos_dates': int((wide_close.index >= OOS_START).sum())})
display(wide_close.tail())


{'price_symbols': 13, 'price_dates': 14109, 'oos_dates': 1627}


,AAPL,AMZN,AVGO,BRK-A,BRK-B,GOOG,GOOGL,LLY,META,MSFT,MU,NVDA,TSLA
date,,,,,,,,,,,,,
2026-06-17,295.9500,237.5000,392.9000,"737,300.0000",491.2800,362.1000,363.7900,"1,112.0000",567.5800,378.9100,"1,043.1900",204.6500,396.3800
2026-06-18,298.0100,244.3900,411.3500,"733,609.7700",489.4600,367.4600,368.0300,"1,098.5700",577.2200,379.4000,"1,133.9900",210.6900,400.4900
2026-06-22,297.0100,232.7900,392.1300,"734,399.9900",488.6900,348.7800,349.6800,"1,102.0800",563.8500,367.3400,"1,211.3800",208.6500,405.0500
2026-06-23,294.3000,234.1100,380.1500,"737,800.0500",492.8100,346.0800,346.1300,"1,107.0800",562.2000,373.9400,"1,051.7700",200.0400,381.6100
2026-06-24,293.0800,234.2700,382.0700,"742,900.0000",494.8100,345.0400,345.2900,"1,117.2600",557.6700,365.4600,"1,048.5100",199.0000,375.5300


## Shared-Book Trading Policy

For every day: exit held positions on the opposite score, then fill open slots from the best current candidates above the entry threshold. Long+short uses one shared `top_k` capacity across both sides.

In [10]:
oos_dates = pd.DatetimeIndex(sorted(set(strategy_scores['date']).intersection(next_returns.index)))
oos_dates = oos_dates[oos_dates >= OOS_START]
trade_symbols = tuple(sorted(set(strategy_scores['symbol']).intersection(next_returns.columns)))
print({'trade_symbols': len(trade_symbols), 'oos_dates': len(oos_dates), 'strategy_sources': strategy_scores['strategy_source'].nunique()})


{'trade_symbols': 13, 'oos_dates': 1627, 'strategy_sources': 16}


In [11]:
weight_artifacts = {}
trade_logs = []
zipline_jobs = []
price_window_start = pd.Timestamp(oos_dates.min()) - pd.Timedelta(days=20)
price_window_end = pd.Timestamp(oos_dates.max())
price_frame_subset = {
    symbol: frame.loc[(pd.DatetimeIndex(frame.index) >= price_window_start) & (pd.DatetimeIndex(frame.index) <= price_window_end)].copy()
    for symbol, frame in price_frames.items()
    if symbol in trade_symbols
}
strategy_source_order = ['ensemble_mean'] + sorted(
    source for source in strategy_scores['strategy_source'].dropna().unique().tolist() if source != 'ensemble_mean'
)

for strategy_source in strategy_source_order:
    score_frame = strategy_scores.loc[strategy_scores['strategy_source'].eq(strategy_source)].copy()
    if score_frame.empty:
        continue
    source_value = score_frame['source'].iloc[0]
    family_value = score_frame['family'].iloc[0]
    print({'strategy_source': strategy_source, 'rows': len(score_frame), 'symbols': score_frame['symbol'].nunique(), 'dates': score_frame['date'].nunique()})
    for variant in ['long_only', 'short_only', 'long_short']:
        for top_k in TOP_K_VALUES:
            weights, trades = build_shared_book_weights(
                score_frame,
                trade_symbols,
                oos_dates,
                top_k=top_k,
                variant=variant,
                entry_threshold=ENTRY_THRESHOLD,
                exit_threshold=EXIT_THRESHOLD,
            )
            weight_artifacts[(strategy_source, variant, top_k)] = weights
            trades = trades.assign(strategy_source=strategy_source, source=source_value, family=family_value, variant=variant, top_k=top_k)
            trade_logs.append(trades)
            zipline_jobs.append(
                ZiplineSharedBookSummaryJob(
                    price_frames=price_frame_subset,
                    target_weights=weights,
                    metadata={
                        'strategy_source': strategy_source,
                        'source': source_value,
                        'family': family_value,
                        'variant': variant,
                        'top_k': top_k,
                        'score_rows': len(score_frame),
                        'score_symbols': score_frame['symbol'].nunique(),
                        'score_dates': score_frame['date'].nunique(),
                        'signal_events': len(trades),
                        'commission_per_share': ZIPLINE_COMMISSION_PER_SHARE,
                        'slippage_bps': ZIPLINE_SLIPPAGE_BPS,
                        'avg_gross_exposure': float(weights.abs().sum(axis=1).mean()),
                        'avg_net_exposure': float(weights.sum(axis=1).mean()),
                        'fully_invested_days': float(weights.abs().sum(axis=1).ge(0.999).mean()),
                        'cash_days': float(weights.abs().sum(axis=1).eq(0).mean()),
                    },
                    capital_base=CAPITAL_BASE,
                    commission_per_share=ZIPLINE_COMMISSION_PER_SHARE,
                    slippage_bps=ZIPLINE_SLIPPAGE_BPS,
                )
            )

print({'zipline_jobs': len(zipline_jobs), 'zipline_max_workers': ZIPLINE_MAX_WORKERS})
backtest_summary = run_zipline_shared_book_summary_jobs(zipline_jobs, max_workers=ZIPLINE_MAX_WORKERS)
trade_log = pd.concat(trade_logs, ignore_index=True) if trade_logs else pd.DataFrame()
backtest_summary = backtest_summary.sort_values(['strategy_source', 'variant', 'top_k']).reset_index(drop=True)
metric_cols = ['framework', 'strategy_source', 'source', 'family', 'variant', 'top_k', 'trades', 'signal_events', 'final_equity', 'total_return', 'annualized_return', 'annualized_vol', 'sharpe', 'max_drawdown', 'avg_gross_exposure', 'avg_net_exposure', 'fully_invested_days', 'cash_days']
display(backtest_summary[metric_cols])
display(trade_log.groupby(['strategy_source', 'variant', 'top_k', 'action'], as_index=False).size().head(80))


{'strategy_source': 'ensemble_mean', 'rows': 21151, 'symbols': 13, 'dates': 1627}


{'strategy_source': 'financetoolkit.ft_growth_balance', 'rows': 21151, 'symbols': 13, 'dates': 1627}


{'strategy_source': 'financetoolkit.ft_growth_cash', 'rows': 21151, 'symbols': 13, 'dates': 1627}


{'strategy_source': 'financetoolkit.ft_growth_income', 'rows': 21151, 'symbols': 13, 'dates': 1627}


{'strategy_source': 'financetoolkit.ft_ratios_efficiency', 'rows': 21151, 'symbols': 13, 'dates': 1627}


...

{'strategy_source': 'fmp.fmp_daily_mcap_yield', 'rows': 21151, 'symbols': 13, 'dates': 1627}


{'strategy_source': 'fmp.fmp_income_mcap', 'rows': 21151, 'symbols': 13, 'dates': 1627}


{'zipline_jobs': 192, 'zipline_max_workers': 4}


,framework,strategy_source,source,family,variant,top_k,trades,signal_events,final_equity,total_return,annualized_return,annualized_vol,sharpe,max_drawdown,avg_gross_exposure,avg_net_exposure,fully_invested_days,cash_days
0,zipline_shared_book_native,ensemble_mean,ensemble,mean,long_only,5,6057,7,"2,340,873.9800",1.3409,0.1408,0.1953,0.7726,-0.2522,1.0000,1.0000,1.0000,0.0000
1,zipline_shared_book_native,ensemble_mean,ensemble,mean,long_only,10,13051,246,"5,145,287.2400",4.1453,0.2888,0.2323,1.2095,-0.3666,0.9622,0.9622,0.7087,0.0000
2,zipline_shared_book_native,ensemble_mean,ensemble,mean,long_only,20,12279,364,"2,596,191.9200",1.5962,0.1592,0.1278,1.2208,-0.2089,0.5010,0.5010,0.0000,0.0000
3,zipline_shared_book_native,ensemble_mean,ensemble,mean,long_only,40,9911,364,"1,625,942.4300",0.6259,0.0782,0.0639,1.2103,-0.1081,0.2505,0.2505,0.0000,0.0000
4,zipline_shared_book_native,ensemble_mean,ensemble,mean,long_short,5,6227,17,"3,472,636.3900",2.4726,0.2127,0.1790,1.1675,-0.2803,1.0000,0.9304,1.0000,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,zipline_shared_book_native,fmp.fmp_income_mcap,fmp,fmp_income_mcap,long_short,40,12404,1589,"740,564.7000",-0.2594,-0.0455,0.0496,-0.9127,-0.2730,0.3250,-0.1038,0.0000,0.0000
188,zipline_shared_book_native,fmp.fmp_income_mcap,fmp,fmp_income_mcap,short_only,5,5827,157,"82,665.6300",-0.9173,-0.3203,0.2801,-1.2380,-0.9278,0.9775,-0.9775,0.9435,0.0000
189,zipline_shared_book_native,fmp.fmp_income_mcap,fmp,fmp_income_mcap,short_only,10,9486,679,"131,826.6300",-0.8682,-0.2694,0.2035,-1.4406,-0.8770,0.8264,-0.8264,0.3577,0.0000
190,zipline_shared_book_native,fmp.fmp_income_mcap,fmp,fmp_income_mcap,short_only,20,9329,793,"357,103.7900",-0.6429,-0.1474,0.1047,-1.4716,-0.6555,0.4288,-0.4288,0.0000,0.0000


,strategy_source,variant,top_k,action,size
0,ensemble_mean,long_only,5,enter_long,6
1,ensemble_mean,long_only,5,exit_long,1
2,ensemble_mean,long_only,10,enter_long,128
3,ensemble_mean,long_only,10,exit_long,118
4,ensemble_mean,long_only,20,enter_long,187
...,...,...,...,...,...
75,financetoolkit.ft_growth_cash,long_short,20,exit_short,4
76,financetoolkit.ft_growth_cash,long_short,40,enter_long,14
77,financetoolkit.ft_growth_cash,long_short,40,enter_short,4
78,financetoolkit.ft_growth_cash,long_short,40,exit_long,1


## Model vs Trading Comparison

In [12]:
model_oos_summary = model_results.loc[model_results['status'].eq('ok'), ['source', 'family', 'oos_rows', 'oos_accuracy', 'oos_balanced_accuracy', 'oos_macro_f1']].copy()
model_oos_summary['strategy_source'] = model_oos_summary.apply(lambda row: strategy_source_name(row['source'], row['family']), axis=1)
display(model_oos_summary.sort_values('oos_macro_f1', ascending=False))

strategy_comparison = (
    backtest_summary.loc[:, ['framework', 'strategy_source', 'source', 'family', 'variant', 'top_k', 'total_return', 'sharpe', 'max_drawdown', 'avg_gross_exposure', 'avg_net_exposure', 'trades']]
    .sort_values(['sharpe', 'total_return'], ascending=False)
    .reset_index(drop=True)
)
display(strategy_comparison)

best_by_strategy_source = (
    strategy_comparison.sort_values(['strategy_source', 'sharpe', 'total_return'], ascending=[True, False, False])
    .groupby('strategy_source', as_index=False)
    .head(1)
    .sort_values(['sharpe', 'total_return'], ascending=False)
    .reset_index(drop=True)
)
display(best_by_strategy_source)

model_vs_trading = best_by_strategy_source.merge(
    model_oos_summary[['strategy_source', 'oos_accuracy', 'oos_balanced_accuracy', 'oos_macro_f1']],
    on='strategy_source',
    how='left',
)
display(model_vs_trading.sort_values(['sharpe', 'total_return'], ascending=False))


,source,family,oos_rows,oos_accuracy,oos_balanced_accuracy,oos_macro_f1,strategy_source
0,financetoolkit,ft_ratios_valuation,3759,0.4302,0.3953,0.3912,financetoolkit.ft_ratios_valuation
1,financetoolkit,ft_ratios_solvency,3759,0.2934,0.3448,0.2676,financetoolkit.ft_ratios_solvency
2,fmp,fmp_cash_mcap,3759,0.2969,0.3642,0.2555,fmp.fmp_cash_mcap
3,financetoolkit,ft_ratios_efficiency,3759,0.2854,0.3410,0.2515,financetoolkit.ft_ratios_efficiency
4,financetoolkit,ft_ratios_profitability,3759,0.2825,0.3402,0.2496,financetoolkit.ft_ratios_profitability
5,fmp,fmp_daily_ev_yield,3759,0.2876,0.3533,0.2486,fmp.fmp_daily_ev_yield
6,fmp,fmp_daily_mcap_yield,3759,0.2961,0.3635,0.2480,fmp.fmp_daily_mcap_yield
7,fmp,fmp_balance_mcap,3759,0.2985,0.3607,0.2423,fmp.fmp_balance_mcap
8,fmp,fmp_income_mcap,3759,0.2966,0.3588,0.2413,fmp.fmp_income_mcap
9,fmp,fmp_daily_ev_multiple,3759,0.2857,0.3524,0.2413,fmp.fmp_daily_ev_multiple


,framework,strategy_source,source,family,variant,top_k,total_return,sharpe,max_drawdown,avg_gross_exposure,avg_net_exposure,trades
0,zipline_shared_book_native,fmp.fmp_daily_mcap_yield,fmp,fmp_daily_mcap_yield,long_only,5,8.1790,1.5984,-0.3022,0.8942,0.8942,6312
1,zipline_shared_book_native,financetoolkit.ft_growth_cash,financetoolkit,ft_growth_cash,long_short,20,2.8823,1.3953,-0.2338,0.6500,0.6396,16885
2,zipline_shared_book_native,financetoolkit.ft_growth_cash,financetoolkit,ft_growth_cash,long_short,40,1.0046,1.3881,-0.1203,0.3250,0.3198,13661
3,zipline_shared_book_native,financetoolkit.ft_growth_balance,financetoolkit,ft_growth_balance,long_short,10,5.2266,1.3796,-0.2792,1.0000,0.9936,13709
4,zipline_shared_book_native,financetoolkit.ft_growth_balance,financetoolkit,ft_growth_balance,long_short,20,2.9116,1.3685,-0.2337,0.6500,0.6450,16927
...,...,...,...,...,...,...,...,...,...,...,...,...
187,zipline_shared_book_native,fmp.fmp_balance_mcap,fmp,fmp_balance_mcap,short_only,10,-0.8602,-1.3014,-0.8735,0.8519,-0.8519,10542
188,zipline_shared_book_native,fmp.fmp_cash_mcap,fmp,fmp_cash_mcap,short_only,5,-0.9404,-1.3517,-0.9490,0.9722,-0.9722,6132
189,zipline_shared_book_native,fmp.fmp_income_mcap,fmp,fmp_income_mcap,short_only,10,-0.8682,-1.4406,-0.8770,0.8264,-0.8264,9486
190,zipline_shared_book_native,fmp.fmp_income_mcap,fmp,fmp_income_mcap,short_only,40,-0.3969,-1.4712,-0.4079,0.2144,-0.2144,7946


,framework,strategy_source,source,family,variant,top_k,total_return,sharpe,max_drawdown,avg_gross_exposure,avg_net_exposure,trades
0,zipline_shared_book_native,fmp.fmp_daily_mcap_yield,fmp,fmp_daily_mcap_yield,long_only,5,8.1790,1.5984,-0.3022,0.8942,0.8942,6312
1,zipline_shared_book_native,financetoolkit.ft_growth_cash,financetoolkit,ft_growth_cash,long_short,20,2.8823,1.3953,-0.2338,0.6500,0.6396,16885
2,zipline_shared_book_native,financetoolkit.ft_growth_balance,financetoolkit,ft_growth_balance,long_short,10,5.2266,1.3796,-0.2792,1.0000,0.9936,13709
3,zipline_shared_book_native,financetoolkit.ft_ratios_efficiency,financetoolkit,ft_ratios_efficiency,long_short,20,2.6490,1.3659,-0.2337,0.6500,0.6157,16914
4,zipline_shared_book_native,financetoolkit.ft_ratios_valuation,financetoolkit,ft_ratios_valuation,long_short,20,2.7878,1.3639,-0.2271,0.6500,0.6271,16905
5,zipline_shared_book_native,financetoolkit.ft_ratios_profitability,financetoolkit,ft_ratios_profitability,long_short,10,5.1249,1.3452,-0.2792,1.0000,0.9936,13727
6,zipline_shared_book_native,financetoolkit.ft_growth_income,financetoolkit,ft_growth_income,long_short,10,4.5442,1.3230,-0.2604,1.0000,0.9899,13597
7,zipline_shared_book_native,financetoolkit.ft_ratios_liquidity,financetoolkit,ft_ratios_liquidity,long_only,20,2.5088,1.3105,-0.2294,0.6163,0.6163,15786
8,zipline_shared_book_native,fmp.fmp_cash_mcap,fmp,fmp_cash_mcap,long_only,5,5.4480,1.3096,-0.3793,0.9496,0.9496,6612
9,zipline_shared_book_native,financetoolkit.ft_ratios_solvency,financetoolkit,ft_ratios_solvency,long_only,20,2.6842,1.3029,-0.2338,0.6456,0.6456,16587


,framework,strategy_source,source,family,variant,top_k,total_return,sharpe,max_drawdown,avg_gross_exposure,avg_net_exposure,trades,oos_accuracy,oos_balanced_accuracy,oos_macro_f1
0,zipline_shared_book_native,fmp.fmp_daily_mcap_yield,fmp,fmp_daily_mcap_yield,long_only,5,8.1790,1.5984,-0.3022,0.8942,0.8942,6312,0.2961,0.3635,0.2480
1,zipline_shared_book_native,financetoolkit.ft_growth_cash,financetoolkit,ft_growth_cash,long_short,20,2.8823,1.3953,-0.2338,0.6500,0.6396,16885,0.2751,0.3372,0.2321
2,zipline_shared_book_native,financetoolkit.ft_growth_balance,financetoolkit,ft_growth_balance,long_short,10,5.2266,1.3796,-0.2792,1.0000,0.9936,13709,0.2700,0.3314,0.2325
3,zipline_shared_book_native,financetoolkit.ft_ratios_efficiency,financetoolkit,ft_ratios_efficiency,long_short,20,2.6490,1.3659,-0.2337,0.6500,0.6157,16914,0.2854,0.3410,0.2515
4,zipline_shared_book_native,financetoolkit.ft_ratios_valuation,financetoolkit,ft_ratios_valuation,long_short,20,2.7878,1.3639,-0.2271,0.6500,0.6271,16905,0.4302,0.3953,0.3912
5,zipline_shared_book_native,financetoolkit.ft_ratios_profitability,financetoolkit,ft_ratios_profitability,long_short,10,5.1249,1.3452,-0.2792,1.0000,0.9936,13727,0.2825,0.3402,0.2496
6,zipline_shared_book_native,financetoolkit.ft_growth_income,financetoolkit,ft_growth_income,long_short,10,4.5442,1.3230,-0.2604,1.0000,0.9899,13597,0.2682,0.3287,0.2272
7,zipline_shared_book_native,financetoolkit.ft_ratios_liquidity,financetoolkit,ft_ratios_liquidity,long_only,20,2.5088,1.3105,-0.2294,0.6163,0.6163,15786,0.2748,0.3356,0.2374
8,zipline_shared_book_native,fmp.fmp_cash_mcap,fmp,fmp_cash_mcap,long_only,5,5.4480,1.3096,-0.3793,0.9496,0.9496,6612,0.2969,0.3642,0.2555
9,zipline_shared_book_native,financetoolkit.ft_ratios_solvency,financetoolkit,ft_ratios_solvency,long_only,20,2.6842,1.3029,-0.2338,0.6456,0.6456,16587,0.2934,0.3448,0.2676


In [13]:
analysis_lines = [
    '## Written Analysis',
    '',
    f'- Universe: {len(symbols)} FMP 1T+ symbols; {len(event_symbols)} had event coverage.',
    f'- Training window: all available rows through {TRAIN_END.date()}. Out-of-sample model and trading window starts {OOS_START.date()}.',
    f'- Trained GPU RF feature-family models: {len(models)}.',
    f'- Strategy sources traded: {strategy_scores["strategy_source"].nunique()} total, including the ensemble mean and {max(strategy_scores["strategy_source"].nunique() - 1, 0)} individual feature-family models.',
    f'- Ensemble OOS prediction rows: {len(mean_scores):,} across {mean_scores["symbol"].nunique()} symbols and {mean_scores["date"].nunique()} dates.',
    f'- Strategy variants: long_only, short_only, long_short with top_k={TOP_K_VALUES}; trade size is fixed at 1/top_k and unused slots stay cash.',
    f'- Execution: native Zipline multi-asset shared-book engine using order_target_percent, ${ZIPLINE_COMMISSION_PER_SHARE:.3f}/share commission, and {ZIPLINE_SLIPPAGE_BPS:.1f} bps slippage.',
]
if not model_oos_summary.empty:
    best_model = model_oos_summary.sort_values('oos_macro_f1', ascending=False).iloc[0]
    analysis_lines.append(f'- Best OOS classifier family: {best_model["strategy_source"]} with macro_f1={best_model["oos_macro_f1"]:.4f}.')
if not backtest_summary.empty:
    best_trade = backtest_summary.sort_values(['sharpe', 'total_return'], ascending=False).iloc[0]
    ensemble_best = backtest_summary.loc[backtest_summary['strategy_source'].eq('ensemble_mean')].sort_values(['sharpe', 'total_return'], ascending=False).head(1)
    single_best = backtest_summary.loc[~backtest_summary['strategy_source'].eq('ensemble_mean')].sort_values(['sharpe', 'total_return'], ascending=False).head(1)
    analysis_lines.extend([
        '',
        'Best native Zipline trading row by Sharpe:',
        f'- {best_trade["strategy_source"]} / {best_trade["variant"]} top_k={int(best_trade["top_k"])}: total_return={best_trade["total_return"]:.2%}, sharpe={best_trade["sharpe"]:.2f}, max_drawdown={best_trade["max_drawdown"]:.2%}, avg_gross={best_trade["avg_gross_exposure"]:.2f}, avg_net={best_trade["avg_net_exposure"]:.2f}.',
    ])
    if not ensemble_best.empty:
        row = ensemble_best.iloc[0]
        analysis_lines.append(f'- Best ensemble_mean row: {row["variant"]} top_k={int(row["top_k"])} with total_return={row["total_return"]:.2%}, sharpe={row["sharpe"]:.2f}, max_drawdown={row["max_drawdown"]:.2%}.')
    if not single_best.empty:
        row = single_best.iloc[0]
        analysis_lines.append(f'- Best individual-model row: {row["strategy_source"]} / {row["variant"]} top_k={int(row["top_k"])} with total_return={row["total_return"]:.2%}, sharpe={row["sharpe"]:.2f}, max_drawdown={row["max_drawdown"]:.2%}.')
    analysis_lines.extend([
        '',
        'Interpretation:',
        '- The per-model strategy variants show whether one feature family is carrying the ensemble or whether averaging adds value.',
        '- Compare model_vs_trading to see whether OOS classifier metrics line up with trading Sharpe; they often will not because ranking, thresholding, and exposure control are separate from multiclass accuracy.',
        '- This notebook uses a native execution engine for the shared multi-asset book instead of single-symbol sleeve runners or cost-only framework rows.',
        '- backtesting.py is intentionally not reported here because it is a single-instrument engine and cannot honestly execute this shared multi-asset book.',
        '- Nautilus shared-book support should be implemented as a native multi-instrument runner before adding Nautilus rows back to this comparison.',
    ])
from IPython.display import Markdown, display
display(Markdown('\n'.join(analysis_lines)))


## Written Analysis

- Universe: 14 FMP 1T+ symbols; 13 had event coverage.
- Training window: all available rows through 2019-12-31. Out-of-sample model and trading window starts 2020-01-01.
- Trained GPU RF feature-family models: 15.
- Strategy sources traded: 16 total, including the ensemble mean and 15 individual feature-family models.
- Ensemble OOS prediction rows: 21,151 across 13 symbols and 1627 dates.
- Strategy variants: long_only, short_only, long_short with top_k=[5, 10, 20, 40]; trade size is fixed at 1/top_k and unused slots stay cash.
- Execution: native Zipline multi-asset shared-book engine using order_target_percent, $0.005/share commission, and 5.0 bps slippage.
- Best OOS classifier family: financetoolkit.ft_ratios_valuation with macro_f1=0.3912.

Best native Zipline trading row by Sharpe:
- fmp.fmp_daily_mcap_yield / long_only top_k=5: total_return=817.90%, sharpe=1.60, max_drawdown=-30.22%, avg_gross=0.89, avg_net=0.89.
- Best ensemble_mean row: long_short top_k=10 with total_return=304.71%, sharpe=1.22, max_drawdown=-31.02%.
- Best individual-model row: fmp.fmp_daily_mcap_yield / long_only top_k=5 with total_return=817.90%, sharpe=1.60, max_drawdown=-30.22%.

Interpretation:
- The per-model strategy variants show whether one feature family is carrying the ensemble or whether averaging adds value.
- Compare model_vs_trading to see whether OOS classifier metrics line up with trading Sharpe; they often will not because ranking, thresholding, and exposure control are separate from multiclass accuracy.
- This notebook uses a native execution engine for the shared multi-asset book instead of single-symbol sleeve runners or cost-only framework rows.
- backtesting.py is intentionally not reported here because it is a single-instrument engine and cannot honestly execute this shared multi-asset book.
- Nautilus shared-book support should be implemented as a native multi-instrument runner before adding Nautilus rows back to this comparison.

## Written Analysis

- Universe: 14 FMP 1T+ symbols; 13 had event coverage.
- Training window: all available rows through 2019-12-31. Out-of-sample model and trading window starts 2020-01-01.
- Trained GPU RF feature-family models: 15.
- Strategy sources traded: 16 total, including the ensemble mean and 15 individual feature-family models.
- Ensemble OOS prediction rows: 21,151 across 13 symbols and 1627 dates.
- Strategy variants: long_only, short_only, long_short with top_k=[5, 10, 20, 40]; trade size is fixed at 1/top_k and unused slots stay cash.
- Execution: native Zipline multi-asset shared-book engine using order_target_percent, $0.005/share commission, and 5.0 bps slippage.
- Best OOS classifier family: financetoolkit.ft_ratios_valuation with macro_f1=0.3912.

Best native Zipline trading row by Sharpe:
- fmp.fmp_daily_mcap_yield / long_only top_k=5: total_return=817.90%, sharpe=1.60, max_drawdown=-30.22%, avg_gross=0.89, avg_net=0.89.
- Best ensemble_mean row: long_short top_k=10 with total_return=304.71%, sharpe=1.22, max_drawdown=-31.02%.
- Best individual-model row: fmp.fmp_daily_mcap_yield / long_only top_k=5 with total_return=817.90%, sharpe=1.60, max_drawdown=-30.22%.

Interpretation:
- The per-model strategy variants show whether one feature family is carrying the ensemble or whether averaging adds value.
- Compare model_vs_trading to see whether OOS classifier metrics line up with trading Sharpe; they often will not because ranking, thresholding, and exposure control are separate from multiclass accuracy.
- This notebook uses a native execution engine for the shared multi-asset book instead of single-symbol sleeve runners or cost-only framework rows.
- backtesting.py is intentionally not reported here because it is a single-instrument engine and cannot honestly execute this shared multi-asset book.
- Nautilus shared-book support should be implemented as a native multi-instrument runner before adding Nautilus rows back to this comparison.